# WavLM Extraction - ALL Audio

Extract from uploaded audio dataset on Kaggle.
GPU: T4
Checkpoint every 25 files.

In [ ]:
# Install
!pip install -q transformers accelerate librosa
import os, json, numpy as np
from tqdm import tqdm
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')

In [ ]:
# Load model
from transformers import WavLMModel
model = WavLMModel.from_pretrained('microsoft/wavlm-base')
model.eval()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print(f'Model on: {device}')

In [ ]:
# Find audio files
AUDIO_DIR = '/kaggle/input/chuckle-audio-all'
OUTPUT_DIR = '/kaggle/working/embeddings'
os.makedirs(OUTPUT_DIR, exist_ok=True)

audio_files = []
for root, dirs, files in os.walk(AUDIO_DIR):
    for f in files:
        if f.endswith(('.wav', '.mp3')):
            vid = f.replace('.wav','').replace('.mp3','')
            audio_files.append((os.path.join(root, f), vid))

print(f'Total audio: {len(audio_files)}')

In [ ]:
# Load checkpoint
CHKPT = '/kaggle/working/checkpoint.json'
if os.path.exists(CHKPT):
    with open(CHKPT, 'r') as f:
        chkpt = json.load(f)
    done = set(chkpt.get('ids', []))
    print(f'Resuming: {len(done)} done')
else:
    done = set()
    print('Starting fresh')

In [ ]:
# Extract
import librosa

def load_audio(path):
    try:
        audio, _ = librosa.load(path, sr=16000, duration=3600)
        return audio
    except: return None

def extract(audio, model, device):
    if audio is None or len(audio) < 320: return None
    embeddings = []
    for i in range(0, len(audio), 480000):  # 30s chunks
        chunk = audio[i:i+480000]
        if len(chunk) < 320: continue
        with torch.no_grad():
            emb = model(torch.FloatTensor(chunk).unsqueeze(0).to(device))
            embeddings.append(emb.last_hidden_state.mean(1).cpu().numpy())
    return np.mean(embeddings, 0) if embeddings else None

results, errors = [], []
for path, vid in tqdm(audio_files):
    if vid in done: continue
    out = f'{OUTPUT_DIR}/{vid}.npy'
    if os.path.exists(out): done.add(vid); continue
    try:
        audio = load_audio(path)
        emb = extract(audio, model, device)
        if emb is not None:
            np.save(out, emb)
            results.append(vid)
            done.add(vid)
    except Exception as e:
        errors.append((vid, str(e)))
    
    # Checkpoint every 25
    if len(results) % 25 == 0:
        with open(CHKPT, 'w') as f:
            json.dump({'ids': list(done), 'results': len(results), 'errors': len(errors)}, f)
        print(f'\nCheckpoint: {len(results)} done, {len(errors)} errors')

print(f'\nDone: {len(results)} extracted, {len(errors)} errors')

In [ ]:
# Final checkpoint
with open(CHKPT, 'w') as f:
    json.dump({'ids': list(done), 'results': len(results), 'errors': len(errors)}, f)
print('Final checkpoint saved')